In [3]:
import numpy as np
from scipy.optimize import minimize

# ==========================================
# Problem parameters
# ==========================================

alpha = 0.25
T = 1
d = 2

l = 0.5
u = 0.6

beta = alpha ** (1 / T)

NGRID = 2000

# ==========================================
# Odd polynomial utilities
# ==========================================

def odd_poly(x, coeffs):
    y = np.zeros_like(x)
    for k, c in enumerate(coeffs):
        y += c * x ** (2 * k + 1)
    return y


def max_error(coeffs, f, interval):
    a, b = interval
    x = np.linspace(a, b, NGRID)
    return np.max(np.abs(f(x) - odd_poly(x, coeffs)))


def minimax_odd_poly(f, interval, d):
    """
    crude L-infinity optimization
    """
    x0 = np.zeros(d + 1)

    obj = lambda c: max_error(c, f, interval)

    res = minimize(
        obj,
        x0,
        method="Powell",
        options={"maxiter": 5000}
    )

    return res.x


# ==========================================
# Stagewise construction
# ==========================================

intervals = [(l, u)]
polys = []

for t in range(T):

    a, b = intervals[-1]

    coeffs = minimax_odd_poly(
        lambda x: x ** beta,
        (a, b),
        d
    )

    polys.append(coeffs)

    xs = np.linspace(a, b, NGRID)
    vals = odd_poly(xs, coeffs)

    intervals.append((vals.min(), vals.max()))

print("Intervals:")
for I in intervals:
    print(I)


# ==========================================
# Composition
# ==========================================

def compose_eval(x):

    y = x.copy()

    for coeffs in polys:
        y = odd_poly(y, coeffs)

    return y


# ==========================================
# Error of composed approximation
# ==========================================

x = np.linspace(l, u, NGRID)

target = x ** alpha

comp_error = np.max(
    np.abs(target - compose_eval(x))
)

print("\nComposed error =", comp_error)


# ==========================================
# Direct approximation
# ==========================================



Intervals:
(0.5, 0.6)
(np.float64(0.7819360499546688), np.float64(0.9390721020920777))

Composed error = 0.05896036529904569


In [4]:
intervals = [(l, u)]
polys = []

for t in range(T-1):

    a, b = intervals[-1]

    coeffs = minimax_odd_poly(
        lambda x: np.ones_like(x),
        (a, b),
        d
    )

    polys.append(coeffs)

    xs = np.linspace(a, b, NGRID)
    vals = odd_poly(xs, coeffs)

    intervals.append((vals.min(), vals.max()))

# final stage approximates x^alpha

a, b = intervals[-1]

coeffs = minimax_odd_poly(
    lambda x: x**alpha,
    (a, b),
    d
)

polys.append(coeffs)

print("Intervals:")
for I in intervals:
    print(I)

Intervals:
(0.5, 0.6)


In [5]:
from math import inf, sqrt
import numpy as np
def optimal_quintic(l, u):
    assert 0 <= l <= u
    if 1 - 5e-6 <= l / u:
# Above this threshold, the equioscillating polynomials
# is numerically equal to...
        return (15/8)/u, (-10/8)/(u**3), (3/8)/(u**5)
# This initialization becomes exact as l -> u
    q = (3*l + u) / 4
    r = (l + 3*u) / 4
    E, old_E = inf, None
    while not old_E or abs(old_E - E) > 1e-15:
        old_E = E
        LHS = np.array([
            [l, l**3, l**5, 1],
            [q, q**3, q**5, -1],
            [r, r**3, r**5, 1],
            [u, u**3, u**5, -1],
        ])
        a, b, c, E = np.linalg.solve(LHS, np.ones(4))
        q, r = np.sqrt((-3*b + np.array([-1, 1]) * sqrt(9*b**2 - 20*a*c)) / (10*c))
    return float(a), float(b), float(c)
def optimal_composition(l, num_iters, cushion=0.02407327424182761):
    u = 1
    coefficients = []
    for _ in range(num_iters):
        a, b, c = optimal_quintic(max(l, cushion*u), u)
# Due to cushioning, this may be centered around 1 with
# respect to 0.024*u, u. Recenter it around 1 with respect
# to l, u, meaning find c so that 1 - c*p(l) = c*p(u) - 1:
        pl = a*l + b*l**3 + c*l**5
        pu = a*u + b*u**3 + c*u**5
        rescalar = 2/(pl + pu)
        a *= rescalar; b *= rescalar; c *= rescalar
# Optionally incorporate safety factor here:
# a /= 1.01; b /= 1.01**3; c /= 1.01**5
        coefficients.append((a, b, c))
        l = a*l + b*l**3 + c*l**5
        u = 2 - l
        print(l,u)
    return coefficients
print(*optimal_composition(1e-3, 10), sep="\n")

0.008287188422276407 1.9917128115777236
0.03403429499099671 1.9659657050090034
0.1342762567262952 1.8657237432737048
0.43958256451702266 1.5604174354829774
0.8764409453036138 1.1235590546963863
0.998815070419226 1.001184929580774
0.9999999989601809 1.0000000010398191
1.0 1.0
1.0 1.0
1.0 1.0
(8.287212018145626, -23.59588651909882, 17.30038731253092)
(4.107059111542195, -2.9478499167379066, 0.5448431082926597)
(3.948690853482296, -2.908902115962949, 0.5518191394370134)
(3.3184196573706006, -2.4884880243148744, 0.5100489401237202)
(2.300652019954819, -1.6689039845747506, 0.41880731195256726)
(1.8913014077873995, -1.2679958271945897, 0.37680408948524946)
(1.8750014808655566, -1.2500016454241956, 0.37500016455956287)
(1.8749999980503391, -1.2499999961006782, 0.3749999980503392)
(1.875, -1.25, 0.375)
(1.875, -1.25, 0.375)


In [6]:
D_total = (2 * d + 1) ** T
d_total = (D_total - 1) // 2

print("Direct odd degree =", D_total)

direct_coeffs = minimax_odd_poly(
    lambda x: x ** alpha,
    (l, u),
    d_total
)

direct_error = np.max(
    np.abs(
        target -
        odd_poly(x, direct_coeffs)
    )
)

print("Direct minimax error =", direct_error)

print("ratio =", comp_error / direct_error)

Direct odd degree = 5
Direct minimax error = 0.05896036529904569
ratio = 1.0


In [29]:
def f(x,y,a,b):
    if (a % 2 == 1) and (b % 2 == 1):
        return x+(y**a - x**b)*0.01
    else:
        return x+(y**(2*a) - x**(2*b))*(0.01*x)

In [41]:
y = 5
a = 1
b=3
x = 3
for i in range(50):
    x = f(x,y,a,b)
x -y**(a/b)

0.006750029871199104